# Part 5 — 평가와 오류 분석

**목적** 점수 하나가 아니라 "무엇을 왜 틀리는가"를 설명한다. ROUGE는 어휘 겹침만 보므로 가이드라인 준수와 사실 오류를 따로 본다.

**대상 모델** Stage C `c_speaker_drop` (pko-t5-base, 전체 278,292건, `speaker/drop`, best = 34,000 스텝)

**실행한 것**
```
python scripts/evaluate_run.py --run artifacts/runs/<c run> --split dev
python scripts/evaluate_run.py --run artifacts/runs/<c run> --split dev --num-beams 1   # greedy 참고
python scripts/evaluate_run.py --run artifacts/runs/<c run> --split dev_debate
python scripts/evaluate_run.py --run artifacts/runs/<c run> --split test                # 최종 1회
```
- `src/guideline_check.py` — 가이드라인 규칙 자동 점검. 정답 요약에도 같은 판정을 적용해 기준선으로 쓴다
- 표본 대조 (현재 Claude 단독 판정) 30건 — `artifacts/eval/human_check_ids.json`에 고정. 판정은 **Claude(AI) 1인**이다
- 이 노트북의 점수는 `scripts/evaluate_run.py`(fp32 생성) 기준이다. 학습 직후 평가(27.325)와 0.13 차이가 있고 원인은 확인하지 못했다

> 2026-09-21 검토: 자동 수치는 검토 신호이며 실제 오류율이 아니다. Claude 판정은 사람 검증을 받지 않았다. 최신 해석·그림은 `../docs/07_full_report.md`와 `../artifacts/report_assets/TEST_ERRORS.md`를 따른다. 기존 실행 출력은 당시 결과다.

In [1]:
import sys, json, yaml
from pathlib import Path
sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_colwidth", 120)
RUNS, EVAL = Path("../artifacts/runs"), Path("../artifacts/eval")
def last(pattern, base=RUNS):
    return sorted(base.glob(pattern))[-1]
def metrics(d):
    return json.loads((d / "metrics.json").read_text(encoding="utf-8"))
dev = last("*_c_speaker_drop_dev", EVAL); dev1 = last("*_c_speaker_drop_dev_beam1", EVAL)
deb = last("*_c_speaker_drop_dev_debate", EVAL); tests = sorted(EVAL.glob("*_c_speaker_drop_test"))
print(dev.name, dev1.name, deb.name, *(t.name for t in tests), sep="\n")

20260920_000117_20260919_172416_c_speaker_drop_dev
20260920_001109_20260919_172416_c_speaker_drop_dev_beam1
20260920_001324_20260919_172416_c_speaker_drop_dev_debate
20260920_001413_20260919_172416_c_speaker_drop_test


## 1. ROUGE — dev (기준별, 디코딩별)

In [2]:
def rouge_rows(d, label):
    m = metrics(d)
    return [{"평가": label, "토큰화": b, "ROUGE-1": s["rouge1"], "ROUGE-2": s["rouge2"], "ROUGE-L": s["rougeL"],
             "ROUGE-L 95% CI": m["rougeL_ci95"][b]["ci95"], "디코딩": m["decoding"], "건수": m["n"]}
            for b, s in m["rouge"].items()]
pd.DataFrame(rouge_rows(dev, "dev") + rouge_rows(dev1, "dev greedy") + rouge_rows(deb, "dev_debate"))

,평가,토큰화,ROUGE-1,ROUGE-2,ROUGE-L,ROUGE-L 95% CI,디코딩,건수
0,dev,space,28.854,13.416,27.456,"[26.468, 28.417]",beam4/nrng0/lp1.0/max64,1500
1,dev,morph,50.652,30.090,45.291,"[44.439, 46.101]",beam4/nrng0/lp1.0/max64,1500
2,dev greedy,space,27.233,11.802,25.859,"[24.895, 26.804]",beam1/nrng0/lp1.0/max64,1500
3,dev greedy,morph,49.685,28.496,44.253,"[43.41, 45.077]",beam1/nrng0/lp1.0/max64,1500
4,dev_debate,space,14.206,5.564,13.527,"[11.517, 15.813]",beam4/nrng0/lp1.0/max64,200
5,dev_debate,morph,38.550,17.678,34.182,"[32.372, 36.155]",beam4/nrng0/lp1.0/max64,200


참고선: 가이드라인 시범 모델 ROUGE-L 19.547 (토큰화 기준이 원문에 없다. 비교가 아니라 참고)

greedy(beam1)는 beam4보다 space 1.6점 낮다. 이후 모든 비교는 beam4/nrng0으로 고정한다.

## 2. 규칙 기반 자동 점검 통과율 — 예측 vs 정답(기준선)

In [3]:
ITEM_KO = {"one_sentence": "한 문장", "no_tag": "태그 없음", "length_ok": "길이 10~150자", "keyword_2plus": "키워드 2개 이상",
           "no_repeat": "어절 3-gram 반복 없음", "ends_da": "'~다' 종결(일상)", "debate_form": "'토론' 표현(토론)", "all_pass": "전 항목"}
def guide(d, label):
    g = metrics(d)["guideline"]
    return pd.DataFrame({f"{label} 예측%": {ITEM_KO[k]: v["pct"] for k, v in g["pred"].items()},
                         f"{label} 정답%": {ITEM_KO[k]: v["pct"] for k, v in g["ref"].items()}})
pd.concat([guide(dev, "dev"), guide(deb, "dev_debate")], axis=1)

,dev 예측%,dev 정답%,dev_debate 예측%,dev_debate 정답%
한 문장,100.00,100.00,100.0,100.0
태그 없음,100.00,100.00,100.0,100.0
길이 10~150자,100.00,100.00,100.0,100.0
키워드 2개 이상,98.27,94.27,99.0,84.0
어절 3-gram 반복 없음,99.73,100.00,98.5,100.0
'~다' 종결(일상),99.87,100.00,NaN,NaN
'토론' 표현(토론),0.00,100.00,4.0,99.5
전 항목,97.33,94.27,4.0,83.5


- dev 전체에서 예측의 자동 점검 통과율(97.3%)이 정답(94.3%)보다 높다. 주요 미통과 항목은 "키워드 2개 이상"이며 다른 항목에도 일부 미통과가 있다.
  가이드라인은 이 조건과 "육하원칙 2요소" 중 하나만 요구한다. 자동 점검으로 정답과의 품질 동등성을 판단할 수 없다.
- **토론 대화는 '토론' 표현이 4.0% (정답 99.5%)**. 낮은 포함률의 원인으로 유형 정보 부재·데이터 불균형을 추정하지만 검증하지 않았다

## 3. 슬라이스 — dev

In [4]:
s = pd.DataFrame(metrics(dev)["slices"]).set_index(["dim", "value"])
print("길이 3분위 경계(train 대화 글자 수):", metrics(dev)["length_tercile_cutoffs_train_chars"])
s

길이 3분위 경계(train 대화 글자 수): [91.0, 130.0]


n  rougeL_space  rougeL_morph  all_pass_pct
dim              value                                                  
length_tercile   김         497        22.332        39.920         97.99
                 중간        461        29.007        46.432         97.83
                 짧음        542        30.835        49.245         96.31
num_participants 2인       1403        27.651        45.461         97.36
                 3인 이상      97        24.626        42.835         96.91
topic            개인 및 관계   381        29.304        47.460         96.85
                 미용과 건강     92        26.557        44.948         96.74
                 상거래(쇼핑)   141        24.544        40.964         97.16
                 시사/교육      70        26.601        44.774         97.14
                 식음료       162        28.403        46.786         96.91
                 여가 생활     189        28.362        44.937         97.88
                 일과 직업     109        27.791        45.752         97.25
                 주거와 생활    242        27.305        45.103         97.11
                 행사        114        23.281        42.407        100.00
type             일상 대화    1492        27.540        45.343         97.86
                 토론 대화       8        11.787        35.533          0.00

## 4. 오류 분류 30건 (Claude 1인 판정)

In [5]:
ea = pd.read_csv(dev / "error_analysis_30.csv", encoding="utf-8-sig")
print(ea["labels"].str.split("|").explode().value_counts().to_string())
print("오류 1개 이상:", (ea["labels"] != "무난").sum(), "/ 30")
print("ROUGE-L 평균 — 무난:", round(ea[ea.labels == "무난"].rougeL_space.mean(), 1), "/ 오류:", round(ea[ea.labels != "무난"].rougeL_space.mean(), 1))
ea[ea.labels != "무난"][["no", "labels", "note", "pred", "ref"]]

labels
무난       19
핵심 누락     6
화자 혼동     3
사실 오류     3
형식 위반     1
오류 1개 이상: 11 / 30
ROUGE-L 평균 — 무난: 28.7 / 오류: 14.1


,no,labels,note,pred,ref
2,3,형식 위반|핵심 누락,일상 대화인데 '~에 대해 토론한다' 형식. 결론(기내용 캐리어)이 빠짐,캐리어를 가져갈 것인지 백팩을 가져갈 것인지에 대해 토론한다.,가을 옷이라 캐리어를 가져 갈 거라고 하면서 제주도 갈 때 하는 기내용으로 가져간다고 한다.
4,5,핵심 누락,결과(집주인에게 문자를 보냄)가 빠지고 앞부분 문답만 요약,도어록이 안돼서 집주인한테 미리 연락해 놓는 게 낫냐고 하자 어차피 집 문이 열려있으니까 그럴 수도 있다고 한다.,동생이 도어록이 안돼서 집주인한테 문자를 보냈다.
7,8,핵심 누락,한 사람의 발화(냉파스타)만 요약. 서로 먹는 것을 묻고 답하는 흐름이 빠짐,마지막 사진은 냉 파스타인데 진짜 맛있다.,서로 무엇을 먹고 있는지에 대해 묻고 답하는 중이다.
11,12,화자 혼동,'내일 또 말해 달라'는 P02 발화인데 P01 발화와 한 주체로 이어 붙임,내일 8시를 잊으면 안 되고 내용 정리는 필요 없을 것 같지만 만약을 대비해서 내일 또 말해달라고 한다.,만약을 대비해 내용 정리를 하고 열심히 해서 백 점을 맞자고 한다.
12,13,사실 오류,추운 원인이 에어컨인데 '날씨가 추워져서 에어컨이 센 것 같다'로 인과가 뒤집힘,갑자기 날씨가 추워져서 에어컨이 센 것 같다.,에어컨으로 인해 떨어진 체감온도에 대해서 대화한다.
14,15,핵심 누락,'그냥 헤어지라'는 핵심이 빠지고 모호한 주제 서술만 남음,저런 말투 때문에 별로라고 한 것에 대해 이야기한다.,그것을 포함해서 말투도 별로라서 그냥 헤어지라고 한다.
16,17,핵심 누락,'아까는 안 산다더니 시간이 생겨 다시 고민' 흐름이 빠짐,5시가 넘어서 사려고 했는데 비싸긴 하지만 이쁘긴 하다.,아까는 있는 것 입겠다며 안 사겠다던 래시가드 비치웨어를 시간 여유가 생겼을 때 살지 말지 다시 생각한다.
18,19,화자 혼동,우산을 두고 온 사람(P01)과 비가 싫다는 사람(P02)을 한 주체로 합침,우산을 놓고 왔는데 비가 와서 싫다고 한다.,"비가 와서 짜증 난다고 하니까, 비가 오면 흐려서 눈이 안 부시지 않냐고 한다."
21,22,화자 혼동|핵심 누락,P01이 직장에서 들은 질문을 두 사람의 문답처럼 서술. '애인'이 정직원 전환된 주체라는 정보가 빠짐,결혼을 언제 할 거냐고 묻자 극보수라고 하며 인턴을 하다가 정직원으로 전환이 된다고 한다.,처음 들어오자마자 받은 질문에 대해 말하고 20대 주임들도 빨리 결혼할 거라고 말한 이야기를 한다.
23,24,사실 오류,지금 운동 중인데 '씻고 누워 운동을 하려고 한다'로 순서·의미가 틀림,아르바이트(알바)가 끝나고 바로 씻고 누워 운동을 하려고 한다.,아르바이트(알바) 끝나고 바로 씻고 누우려다가 마음먹고 운동을 했다.


- **사실 오류 3/30**: 인과 뒤집힘, 상태·순서 오류, 추측을 단정 → **"최종 모델 확정"이라 쓰지 않는다** (CLAUDE.md 9장)
- **화자 혼동 3/30**: 두 사람의 발화를 한 주체로 이어 붙인다 (3건 모두 2인 대화)
- **핵심 누락 6/30**: 앞부분·한 사람 발화만 요약하고 결론을 놓친다
- ROUGE-L 0인데 무난으로 판정한 건이 3건 있다. ROUGE는 정답과 다른 부분을 고른 요약을 낮게 본다
- 30건 표본이라 비율의 불확실성이 크다 (사실 오류 10%, 대략 2~27%)

## 5. 최종 test (공식 valid 전체, 1회)

In [6]:
if tests and (tests[-1] / "metrics.json").exists():
    t = tests[-1]; mt = metrics(t)
    display(pd.DataFrame(rouge_rows(t, "test")))
    display(guide(t, "test"))
    display(pd.DataFrame(mt["slices"]).set_index(["dim", "value"]))
else:
    print("test 평가가 아직 끝나지 않았다")

,평가,토큰화,ROUGE-1,ROUGE-2,ROUGE-L,ROUGE-L 95% CI,디코딩,건수
0,test,space,28.414,13.036,27.154,"[26.949, 27.358]",beam4/nrng0/lp1.0/max64,35004
1,test,morph,50.652,29.888,45.354,"[45.16, 45.537]",beam4/nrng0/lp1.0/max64,35004


,test 예측%,test 정답%
한 문장,99.99,99.98
태그 없음,100.00,100.00
길이 10~150자,100.00,100.00
키워드 2개 이상,98.16,95.35
어절 3-gram 반복 없음,99.43,99.88
'~다' 종결(일상),99.88,100.00
'토론' 표현(토론),4.79,100.00
전 항목,97.02,95.21


n  rougeL_space  rougeL_morph  all_pass_pct
dim              value                                                   
length_tercile   김        11682        23.271        41.071         97.55
                 중간       11698        27.333        45.497         97.67
                 짧음       11624        30.877        49.513         95.84
num_participants 2인       32642        27.226        45.402         97.03
                 3인 이상     2362        26.168        44.688         96.99
topic            개인 및 관계   8892        27.437        45.554         96.46
                 미용과 건강    2134        27.575        45.287         98.17
                 상거래(쇼핑)   3288        25.938        43.943         96.78
                 시사/교육     1640        27.581        45.902         97.87
                 식음료       3773        27.731        46.442         96.50
                 여가 생활     4407        25.956        44.502         97.25
                 일과 직업     2554        27.046        45.345         98.08
                 주거와 생활    5648        27.605        45.654         97.08
                 행사        2668        27.423        45.378         97.00
type             일상 대화    34816        27.214        45.402         97.52
                 토론 대화      188        16.020        36.458          4.26

## 6. 결론과 다음 후보

- 최종 후보: Stage C `c_speaker_drop`. test 결과는 위 표, 요약은 `artifacts/eval/<test 폴더>/RESULT.md`
- 남은 문제와 다음 실험 후보 (한 번에 하나씩):
  1. 토론 형식 — 유형 표시를 이미 dev에서 실험했으며 문자열 포함률 1.5%→19.5%를 관찰했다. 최종 후보에는 채택하지 않았다
  2. 사실 오류·화자 혼동 — 100건으로 확대했으나 Claude 단독 판정이다. 사람 독립 판정과 일치도를 확인해야 한다
  3. 수렴 — 3 epoch 실험에서는 추가 이득을 확인하지 못했다. 수렴 여부를 확정한 것은 아니다